In [2]:
import gradio as gr
import json
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader

/Users/tkarmysh/tlnt-ai/talantasan-git/agents/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv(override=True)
openai = OpenAI()

In [4]:
def record_user_details(email: str, name: str = "Not provided", notes: str = "Not provided"):
    with open("user_details.json", "w") as f:
        print(f"Function called, to record email: {email}")
        f.write(json.dumps({"email": email, "name": name, "notes": notes}) + "\n")
        

def record_unknown_question(question: str):
    with open("unknown_questions.json", "a") as f:
        print(f"Irrelevant questioin : {question}, is recorded to the file")
        f.write(json.dumps({"question": question}) + "\n")

In [5]:
record_user_details("talant@example.com", "Talant", "This is a test user.")

Function called, to record email: talant@example.com


In [6]:
record_unknown_question("What is the capital of France?")

Irrelevant questioin : What is the capital of France?, is recorded to the file


In [7]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Records user details in a JSON file.",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of the user."},
            "name": {"type": "string", "description": "The name of the user."},
            "notes": {"type": "string", "description": "Additional notes about the user."}
        },
        "required": ["email"]
    },
}

record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Records unknown questions in a JSON file.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string", 
                "description": "Always use this tool to record any question that couldn't be answed as you don't know the answer"
                }
        },
        "required": ["question"]
    },
}

In [8]:
tools = [
    {"type": "function", "function": record_user_details_json},
    {"type": "function", "function": record_unknown_question_json}
]

In [17]:
def handle_tools_calls_if(tool_calls):
    results = []
    for call in tool_calls:
        tool_name = call.function.name
        arguments = json.loads(call.function.arguments)
        print(f"Tool called: {tool_name} with arguments: {arguments}")
        
        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)
        
        results.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": call.id
        })
    return results        

In [18]:
def handle_tools_calls_globals(tool_calls):
    results = []
    for call in tool_calls:
        tool_name = call.function.name
        arguments = json.loads(call.function.arguments)
        print(f"Tool called: {tool_name} with arguments: {arguments}")
        
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {"Error": "No tool found"}
        
        results.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": call.id 
        })
    return results

In [19]:
def linkedin_profile(file_path):
    reader = PdfReader(file_path)
    linkedin = ""
    for page in reader.pages:
        linkedin += page.extract_text()
    return linkedin

In [20]:
linkedin_text = linkedin_profile("docs/linkedin.pdf")

In [24]:
system_prompt = f"""
You are a digital twin of the person whose LinkedIn profile is provided.
You have access to the following LinkedIn profile text:
{linkedin_text}

# Rules
Engage with the user.
Act like a digital twin of this person. 
Respond to the questions and prompts as if you are this person.
Stay true to the information provided in the LinkedIn profile.
Do not provide information that is not present in the LinkedIn profile.
If you don't find information in the LinkedIn profile, respond with "I don't have that information."

IMPORTANT!
if irrelevant question is asked, ask user if they are interested to learn more about this person, if yes ask their email and record using tool.
Use the tool to store the email address provided by the user.

Also if the user asks the question that is not relevant store this question using the tool.
"""

In [25]:
def chat_history(message, history):
    messages = [{"role": "system", "content": system_prompt,}] + history + [{"role": "user", "content": message}]
    
    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=messages,
        tools=tools
    )
    
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        result = handle_tools_calls_if(tool_calls)
        messages.append(message)
        messages.extend(result)
        response = openai.chat.completions.create(
            model="gpt-5.4-nano",
            messages=messages,
            tools=tools
        )
    
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat_history).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Tool called: record_unknown_question with arguments: {'question': 'What is your favorite drink?'}
Irrelevant questioin : What is your favorite drink?, is recorded to the file
Tool called: record_user_details with arguments: {'email': 'sam@test.py', 'notes': 'User provided email in response to offer to learn more; no additional context given.'}
Function called, to record email: sam@test.py
